# 03 — Run experiments

Configure and launch experiments from a **single configuration cell** below.

Each run writes under the configured storage root (`outputs/runs/` locally, `UH_CV/runs/` on Google Drive).

**Prerequisite:** notebook 02 → `data/annotations/validation_lengths.csv`

In [2]:
from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments import load_validation_image_ids
from src.experiments.notebook_helpers import (
    RunExperimentsConfig,
    build_experiment_specs,
    preview_experiment_specs,
    project_config_for_experiments,
    resolve_runs_root,
    run_configured_experiments,
)
from src.utils import setup_logging

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
setup_logging()
cfg = get_config()
print(f"Storage: {STORAGE} | Colab: {cfg.is_colab}")
VALIDATION_IDS = load_validation_image_ids(cfg)
print(f"Validation set: {len(VALIDATION_IDS)} images")

ModuleNotFoundError: No module named 'src'

## 2×2 comparison (4 runs): ground truth × {skeleton, regression}

| Ground truth CSV | Skeleton baseline (no regression) | Regression calibrated |
|------------------|-----------------------------------|------------------------|
| `validation_lengths.csv` | `baseline_skeleton_validation_lengths` | `regression_skeleton_validation_lengths` |
| `validation_lengths2.csv` (fixed image **2572**) | `baseline_skeleton_validation_lengths2` | `regression_skeleton_validation_lengths2` |

**Latest local results** (valid split, n=30) — see `outputs/runs/comparison_grid_summary.csv`:

| GT | Method | MAE (mm) | RMSE (mm) |
|----|--------|----------|-----------|
| validation_lengths | skeleton | 51.7 | 125.7 |
| validation_lengths | regression | 22.8 | 39.3 |
| validation_lengths2 | skeleton | 31.2 | 44.8 |
| validation_lengths2 | regression | 23.9 | 45.4 |

`validation_lengths` still labels 2572 as **1090 mm**; `validation_lengths2` uses **640 mm**, which is why baseline RMSE drops sharply on v2.

### Colab / notebook flags (`RUN_CFG` in the next code cell)

| Setting | Purpose |
|---------|---------|
| `ground_truth_source="validation_lengths"` or `"validation_lengths2"` | Which CSV under `data/annotations/` |
| `splits=["valid"]` | **Required** — validation IDs live in `images/valid`, not `test` |
| `run_regression_calibration=True` | Train a **new** model + run (`regression_calibrated_v1` row in preview) |
| `use_regression_model=True` + `regression_model_path=...` | **Reuse** a saved `.joblib` on baseline runs (no training) |

### Where artifacts are saved

Each run: `outputs/runs/<run_name>/` (or Drive equivalent via `setup_notebook_environment`)

- `predictions.csv` — final lengths (regression run also has `skeleton_length_mm` column)
- `comparison.csv` — per-image errors vs ground truth
- `metrics.json` — MAE / RMSE (regression runs include `skeleton_baseline_*` too)
- `regression_model.joblib` — **only** under `regression_skeleton_<ground_truth_source>/`

### Re-run all four cells in one shot (optional cell below)

```python
from src.experiments import run_four_way_comparison, format_comparison_table
summary = run_four_way_comparison(exp_cfg, overwrite=True)
print(format_comparison_table(summary))
```


## Configuration

Edit this cell only. Set `RUN = True` to execute; leave `False` to preview the experiment grid without running.

Use `experiments=[...]` for explicit runs (legacy mode). Otherwise a grid is built from `pipelines` × `methods` × `splits`.

In [ ]:
RUN = True  # set False to preview specs without executing

RUN_CFG = RunExperimentsConfig(
    # --- scope ---
    pipelines=["baseline"],            # "baseline" | "advanced"
    methods=["skeleton"],
    splits=["valid"],
    image_ids=None,                    # explicit list overrides validation_set_only
    validation_set_only=True,
    ground_truth_source="validation_lengths2",  # validation_lengths | validation_lengths2 | lengths_mm
    limit=None,

    # --- execution ---
    overwrite=True,
    visualize=True,                   # save debug figures under run_dir/figures/
    evaluate=True,
    runs_root=None,                   # default: cfg.runs_root (Drive or outputs/runs)
    run_name_template="{pipeline}_{method}_grid_v1",

    # --- Colab / Drive persistence (all default True) ---
    cache_results=True,
    save_figures_to_drive=True,
    save_metrics_to_drive=True,
    save_predictions_to_drive=True,

    # --- advanced pipeline flags (when pipeline="advanced") ---
    perspective=False,
    use_grid_auto_calibration=True,
    use_depth_estimation=False,
    use_3d_measurement=False,

    # --- regression calibration (notebook 02 labels required) ---
    # train_regression: set True to add a "regression" row in preview + train a model
    # apply_saved_model: set use_regression_model=True + regression_model_path for baseline
    run_regression_calibration=False,  # True = train NEW model (see markdown above)
    use_regression_model=False,        # True = apply saved model at regression_model_path
    regression_model_path=None,        # e.g. REPO / "outputs/runs/regression_skeleton_validation_lengths2/regression_model.joblib"
    regression_run_name="regression_calibrated_v1",
    regression_model_type="random_forest",  # or "xgboost" if installed
    regression_train_split="valid",
    use_regression_model=False,
    regression_model_path=None,

    # --- optional: explicit experiment list (overrides grid above) ---
    experiments=None,
    # experiments=[
    #     {"pipeline": "advanced", "method": "skeleton", "run_name": "advanced_full_v2",
    #      "split": "valid", "validation_set_only": True, "overwrite": True,
    #      "use_grid_auto_calibration": True, "use_depth_estimation": True,
    #      "use_3d_measurement": True},
    # ],
)

SPECS = build_experiment_specs(RUN_CFG)
preview_experiment_specs(SPECS, RUN_CFG)

## Run experiments

In [ ]:
# Optional: run full 2×2 grid (both GT files × skeleton + regression). ~15–25 min on Colab.
RUN_FOUR_WAY = False

if RUN_FOUR_WAY:
    from src.experiments import run_four_way_comparison, format_comparison_table

    grid = run_four_way_comparison(project_config_for_experiments(RUN_CFG), overwrite=True)
    display(grid)
    print(format_comparison_table(grid))


In [ ]:
import pandas as pd

exp_cfg = project_config_for_experiments(RUN_CFG)
runs_root = resolve_runs_root(RUN_CFG)

if RUN:
    summary = run_configured_experiments(RUN_CFG, SPECS, cfg=exp_cfg)
else:
    print("RUN=False — skipped execution. Set RUN=True in the configuration cell.")
    summary = pd.DataFrame()

if len(summary):
    display(summary[[c for c in ["run_name", "pipeline", "method", "n_predictions", "mae_mm", "rmse_mm", "run_dir"] if c in summary.columns]])

## Sanity check

Confirms every annotated fish was predicted and evaluated when using the validation set.

In [ ]:
if len(summary) and RUN_CFG.validation_set_only and RUN_CFG.image_ids is None:
    n_gt = len(VALIDATION_IDS)
    for _, row in summary.iterrows():
        pred_path = Path(row["predictions_path"])
        comp_path = Path(row["comparison_path"]) if pd.notna(row.get("comparison_path")) else None
        pred = pd.read_csv(pred_path)
        comp = pd.read_csv(comp_path) if comp_path and comp_path.is_file() else None
        print(f"{row['run_name']}: predictions={len(pred)}", end="")
        if comp is not None:
            print(f"  evaluated={len(comp)}", end="")
            assert len(pred) == n_gt, f"{row['run_name']}: not all validation IDs predicted"
            assert len(comp) == n_gt, f"{row['run_name']}: evaluation incomplete"
            print("  OK")
        else:
            print("  (no comparison.csv)")
else:
    print("Skipped — custom image_ids or no runs executed.")